In [1]:
import json
import numpy as np
import os

# 경로 설정
EXPERT_PATH = r"D:\kpop-dance-dtw-analysis\dtw-analysis\data\json\expert_lovedive_visible.json"

# 데이터 로드
with open(EXPERT_PATH, "r", encoding="utf-8") as f:
    expert_data = json.load(f)

print(f"✅ 데이터 로드 완료")
print(f"총 프레임 수: {len(expert_data['frames'])}")

✅ 데이터 로드 완료
총 프레임 수: 4369


In [2]:
# MediaPipe 관절 ID
# 23 = 왼쪽 골반, 24 = 오른쪽 골반

def normalize_landmarks(landmarks):
    """
    골반 중심을 원점(0,0)으로 정규화
    - 체형, 카메라 거리 차이 제거
    - 동작 자체만 비교 가능하게 변환
    """
    if landmarks is None:
        return None

    # 골반 중심 계산
    left_hip = next((lm for lm in landmarks if lm["id"] == 23), None)
    right_hip = next((lm for lm in landmarks if lm["id"] == 24), None)

    if left_hip is None or right_hip is None:
        return None

    # 골반 중심점 (원점)
    hip_x = (left_hip["x"] + right_hip["x"]) / 2
    hip_y = (left_hip["y"] + right_hip["y"]) / 2

    # 어깨 너비로 스케일 정규화 (id=11: 왼쪽 어깨, id=12: 오른쪽 어깨)
    left_shoulder = next((lm for lm in landmarks if lm["id"] == 11), None)
    right_shoulder = next((lm for lm in landmarks if lm["id"] == 12), None)

    if left_shoulder and right_shoulder:
        scale = abs(left_shoulder["x"] - right_shoulder["x"])
        scale = scale if scale > 0 else 1.0
    else:
        scale = 1.0

    # visible=True 관절만 정규화
    normalized = []
    for lm in landmarks:
        if not lm["visible"]:
            continue
        normalized.append({
            "id": lm["id"],
            "x": round((lm["x"] - hip_x) / scale, 6),
            "y": round((lm["y"] - hip_y) / scale, 6),
            "visibility": lm["visibility"]
        })

    return normalized

print("✅ 정규화 함수 정의 완료")

✅ 정규화 함수 정의 완료


In [3]:
def build_normalized_sequence(data, fixed_ids=None):
    """
    fixed_ids: 항상 사용할 관절 ID 고정
    → 프레임마다 관절 수가 달라지는 문제 해결
    """
    # 주요 관절 33개 중 핵심 11개만 고정 사용
    if fixed_ids is None:
        fixed_ids = [0, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26]
        # 코, 양쪽어깨, 팔꿈치, 손목, 골반, 무릎

    sequence = []

    for frame in data["frames"]:
        normalized = normalize_landmarks(frame["landmarks"])

        if normalized is None or len(normalized) == 0:
            continue

        # id 기준 딕셔너리로 변환
        lm_dict = {lm["id"]: lm for lm in normalized}

        # 고정 관절만 순서대로 추출
        coords = []
        valid = True
        for fid in fixed_ids:
            if fid not in lm_dict:
                valid = False
                break
            coords.extend([lm_dict[fid]["x"], lm_dict[fid]["y"]])

        if valid:
            sequence.append(coords)

    return np.array(sequence)

# 전문가 시퀀스 생성
expert_seq = build_normalized_sequence(expert_data)

print(f"✅ 정규화 완료")
print(f"전문가 시퀀스 shape: {expert_seq.shape}")
print(f"  → (프레임 수, 관절좌표 수)")

✅ 정규화 완료
전문가 시퀀스 shape: (2720, 22)
  → (프레임 수, 관절좌표 수)


In [4]:
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
print("✅ fastdtw 사용 가능")

✅ fastdtw 사용 가능


In [5]:
# 전문가 시퀀스를 절반으로 나눠서 테스트
seq_a = expert_seq[:500]   # 앞 500프레임
seq_b = expert_seq[500:1000]  # 뒤 500프레임

print(f"시퀀스 A shape: {seq_a.shape}")
print(f"시퀀스 B shape: {seq_b.shape}")

# DTW 계산
print("\n⏳ DTW 계산 중...")
distance, path = fastdtw(seq_a, seq_b, dist=euclidean)

print(f"\n✅ DTW 계산 완료!")
print(f"DTW Distance: {distance:.4f}")
print(f"Warping Path 길이: {len(path)}")

시퀀스 A shape: (500, 22)
시퀀스 B shape: (500, 22)

⏳ DTW 계산 중...

✅ DTW 계산 완료!
DTW Distance: 2233.6459
Warping Path 길이: 561


In [6]:
def dtw_to_score(distance, max_distance=5000):
    """
    DTW Distance → 0~100점 유사도 점수 변환
    - distance=0 → 100점 (완벽히 일치)
    - distance=max_distance 이상 → 0점
    """
    score = max(0, 100 - (distance / max_distance * 100))
    return round(score, 2)

score = dtw_to_score(distance)
print(f"✅ 유사도 점수: {score}점 / 100점")
print(f"  (전문가 앞부분 vs 뒷부분 비교)")

✅ 유사도 점수: 55.33점 / 100점
  (전문가 앞부분 vs 뒷부분 비교)


In [7]:
# 같은 구간끼리 비교 → 100점 나와야 정상
seq_same_a = expert_seq[:500]
seq_same_b = expert_seq[:500]

distance_same, _ = fastdtw(seq_same_a, seq_same_b, dist=euclidean)
score_same = dtw_to_score(distance_same)

print(f"✅ 동일 구간 비교 검증")
print(f"DTW Distance: {distance_same:.4f}")
print(f"유사도 점수: {score_same}점 / 100점")
print(f"→ 100점에 가까울수록 점수 체계 정상!")

✅ 동일 구간 비교 검증
DTW Distance: 0.0000
유사도 점수: 100.0점 / 100점
→ 100점에 가까울수록 점수 체계 정상!
